# The shape of problems

> Before any model: what kind of question is this, what feedback exists, and how to notice when the honest answer is that no model can help.

Read this chapter at `/learn/03-the-shape-of-problems/`. Exported from `src/content/chapters/03-the-shape-of-problems.mdx` — edit there, not here.


Yesterday you learned the tools. Today, the thing that actually separates people
who ship models from people who train them: deciding what the problem *is*.

This is the least glamorous chapter here and the one with the highest return. A
mediocre model on a well-framed problem beats an excellent model on a badly
framed one, every time, and the second failure is invisible until it is
expensive.

## Three independent questions

Every project answers all three of these. They are orthogonal — knowing one tells
you nothing about the others — and conflating them is the main reason the field's
vocabulary feels like soup.

<div class="table-scroll">

| | The question | The answers |
|---|---|---|
| **1. Supervision** | What feedback do I get? | supervised · unsupervised · self-supervised · reinforcement |
| **2. Task** | What shape is the output? | classification · regression · ranking · generation · clustering |
| **3. Model family** | What shape is the function? | linear · tree · neural network · … |

</div>

So "supervised classification with a gradient-boosted tree" is one answer from
each column. Not a single thing with a long name.

Question 1 decides what data you must go and collect, and that is the expensive
irreversible decision. Question 3 is a one-line change you can revisit on a
Thursday afternoon. Spend your thinking accordingly.

## Question 1: what feedback do you get?

**Supervised** means every example carries the right answer. You have 50,000
emails and, for each, a human-supplied "spam" or "not spam". This is where nearly
all commercially deployed machine learning lives, because it is the setting where
you can actually measure whether you are winning.

The catch is that labels cost money. A radiologist labelling scans is a
radiologist not doing radiology. Half of applied machine learning is the question
"how do I get labels cheaply enough" — and the answers include buying them,
crowdsourcing them, mining them from logs you already have, or the next category.

**Self-supervised** is the trick that ate the world: hide part of the input and
train the model to predict it. No human labels the data because the data labels
itself. Take any sentence, remove the last word, and you have a training example.
The whole internet is now a labelled dataset.

In [ ]:
text = "the quick brown fox jumps over the lazy dog"
tokens = text.split()
pairs = [(tokens[:i], tokens[i]) for i in range(1, len(tokens))]
for context, target in pairs[:4]:
    print(f"{' '.join(context):32s} -> {target}")

That is the training objective of every large language model, in its entirety.
The sophistication is all in the model and the scale; the supervision signal is
this. It is the single most important idea in the last decade, and the reason it
matters is economic rather than mathematical — it made data free.

**Unsupervised** means no labels at all, and no way to manufacture them. You have
a million customer records and no idea what the interesting groupings are.
Clustering, dimensionality reduction and anomaly detection live here. The
uncomfortable property is that there is no score: nothing tells you whether your
clusters are *right*, only whether they are useful to a human downstream. Budget
for that ambiguity before you promise anyone a number.

**Reinforcement learning** means feedback that is delayed and evaluative rather
than instructive. You are not told the correct move; you are told, forty moves
later, that you lost. It is how game-playing agents and robot controllers are
trained, and — as RLHF — how a raw language model is turned into something that
answers your question instead of continuing your sentence.

Reinforcement learning is deliberately out of scope here. It is a genuinely
separate discipline with its own vocabulary (states, actions, policies, value
functions, the Bellman equation) and its own failure modes, and two weeks cannot
hold both. What you need for now is to recognise the shape: *delayed,
evaluative, and the agent's own actions determine what data it sees next.* That
last clause is what makes it hard.

## Question 2: what shape is the output?

Given supervision, the output shape determines your loss function and your
metric — which is to say, it determines what "good" means.

<div class="table-scroll">

| Task | Output | Loss you will use | Example |
|---|---|---|---|
| Binary classification | one of two labels | binary cross-entropy | fraud / not fraud |
| Multi-class | one of k labels | cross-entropy | which of 37 breeds |
| Multi-label | any subset of k | binary cross-entropy per label | tags on a photo |
| Regression | a number | MSE, MAE, Huber | house price |
| Ordinal | an ordered grade | depends; often regression | 1–5 star rating |
| Ranking | an ordering | pairwise / listwise losses | search results |
| Generation | a sequence | cross-entropy per token | translation, text |

</div>

The row people get wrong is **multi-label**. "Which breed is this dog" has one
answer and wants softmax, where the probabilities compete and sum to one. "Which
of these tags apply to this photo" can have three answers, and forcing them to
compete is simply the wrong model of the world. It is one line of code — sigmoid
per label instead of one softmax — and it is a substantial accuracy difference.

The row people underestimate is **ordinal**. A 1-star and a 5-star review are not
merely two different classes; being wrong by four stars is worse than being wrong
by one, and plain classification does not know that.

## Question 3: features and targets

Now the concrete part. A supervised dataset is a table where you have chosen one
column to predict.

In [ ]:
import numpy as np, pandas as pd

df = pd.DataFrame({
    "sqm":        [45, 62, 80, 55, 120, 95, 38, 70],
    "bedrooms":   [1, 2, 3, 2, 4, 3, 1, 2],
    "district":   ["c", "n", "n", "c", "s", "s", "c", "n"],
    "price_eur":  [220, 310, 395, 265, 610, 470, 190, 340],
})
df

- **Features** (`X`) — everything you are allowed to look at. Conventionally
  capital, because it is a matrix of shape `(n_samples, n_features)`.
- **Target** (`y`) — the thing you are predicting. Lowercase, because it is a
  vector.
- **A sample** — one row. Also called an example, an instance, or an observation,
  depending on which decade the author learned this.

In [ ]:
X = df[["sqm", "bedrooms"]].values      # numeric features only, for now
y = df["price_eur"].values
X.shape, y.shape

Note `X.shape` is `(8, 2)` and `y.shape` is `(8,)`. Nearly every shape error you
hit in the next fortnight is `X` accidentally being one-dimensional, or `y`
accidentally being `(8, 1)`. Print them.

### The `district` column

`district` is a string, and models consume numbers. The naive fix is to map
`c → 0, n → 1, s → 2`, and it is wrong — it tells the model that south is three
times north and that north sits *between* centre and south. You have invented an
ordering that does not exist.

In [ ]:
onehot = pd.get_dummies(df["district"], prefix="d", dtype=int)
pd.concat([df[["sqm"]], onehot], axis=1).head(4)

One column per category, exactly one of them hot. No spurious ordering. The cost
is width: 50,000 distinct user IDs become 50,000 columns, which is untenable —
and the fix for that is **embeddings**, which are one of the genuinely beautiful
ideas in the field and are [Chapter 12](/learn/12-embeddings-and-tabular/).

One-hot encoding is exactly a fieldless `enum` lowered to its discriminant, then
widened so no arithmetic relationship between variants is implied. An embedding
is what you would build if you decided each variant should carry a small learned
`[f32; 16]` payload, and let training decide what goes in it.

### Leakage: the mistake that feels like success

In [ ]:
from sklearn.linear_model import LinearRegression

leaky = df.copy()
leaky["price_per_sqm"] = leaky["price_eur"] / leaky["sqm"]   # <- uses the target

Xl = leaky[["sqm", "price_per_sqm"]].values
model = LinearRegression().fit(Xl, y)
print(f"R^2 = {model.score(Xl, y):.4f}   (suspiciously perfect)")

`price_per_sqm` was computed *from the price*. The model has been handed the
answer in a thin disguise, and it scores perfectly in testing and catastrophically
in production, where nobody knows the price yet — that being the entire reason
you built it.

**Leakage** is any information in your features that would not be available at
prediction time. It is the single most common way real projects fail, it always
presents as unusually good results, and it is very rarely as obvious as this.

Real leakage looks like: a `last_updated` timestamp that only gets written when a
case is resolved. A patient ID that encodes which hospital, where one hospital
only sees severe cases. An "account status" field back-filled nightly. A row
duplicated between your training and validation sets.

The habit that catches it: when a result is much better than you expected, do not
celebrate. Go and find out why. It is leakage far more often than it is genius.

## How to interrogate a dataset

Before modelling, always. Fifteen minutes, in this order.

In [ ]:
print(df.dtypes.to_dict()); print()
print(df.describe().round(1))

In [ ]:
import numpy as np
labels = np.array([0]*950 + [1]*50)          # a realistic fraud-ish target
print("class balance:", np.bincount(labels))
print(f"always-predict-0 accuracy: {(labels == 0).mean():.1%}")

Ninety-five percent accuracy, from a model that is a constant. If somebody quotes
you an accuracy without a class balance, they have told you nothing. This is why
[metrics](/learn/16-shipping-and-reading-papers/) get a chapter of their own, and
why precision and recall exist.

In [ ]:
print(df.groupby("district")["price_eur"].agg(["mean", "count"]))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.scatter(df["sqm"], df["price_eur"])
plt.xlabel("sqm"); plt.ylabel("price (k EUR)"); plt.title("always plot it")
plt.tight_layout()

Step 4 is not optional. Anscombe's quartet — four datasets with identical means,
variances and correlations, and wildly different shapes — exists to make exactly
this point. Summary statistics can be identical while the data is not remotely
similar.

## Is this even a machine learning problem?

Four questions, in order. A "no" to any of them means stop.

**1. Could a competent human do it from the same information?**
If a person staring at the same row cannot tell, the signal is probably not
there. This is not a law — models find patterns humans cannot, particularly in
high dimensions — but as a first filter it is excellent, and it will save you
from most doomed projects.

**2. Do I have enough examples, of the thing I care about?**
Not rows — *examples of the positive class*. A million transactions with eleven
frauds is eleven examples. The rough floors: hundreds per class for tabular
problems, a few dozen per class for images if you can start from a pretrained
model ([Chapter 11](/learn/11-vision-and-transfer/)), and thousands if you are
training from scratch.

**3. Will the future resemble the past?**
Models learn the distribution they were shown. If your training data is
pre-pandemic travel behaviour, or pre-competitor pricing, the model is a
historian. This is *distribution shift*, and it is why models decay in production
rather than staying fixed.

**4. What happens when it is wrong?**
Not rhetorical. Write down the cost of a false positive and the cost of a false
negative, in whatever units your organisation cares about. Those two numbers
determine your metric, your decision threshold, and whether the project is worth
doing. Skipping this is how teams optimise accuracy for six months on a problem
where recall was the only thing that mattered.

Frame the problem, then check that a trivial baseline does not already solve it,
then model. The baseline — predict the mean, predict the majority class, use last
week's value — takes ten minutes and is the number every later result must beat
to mean anything.

## Exercise

Frame these three. For each, name the supervision, the task, the features, the
target, one plausible source of leakage, and a trivial baseline. Then decide
whether it is a machine learning problem at all.

1. Predict which of your users will cancel their subscription next month.
2. Decide which of 4 million support tickets are about the same underlying bug.
3. Compute VAT owed on an invoice.

**1. Churn.** Supervised binary classification. Features: usage in the last 30
days, tenure, support contacts, plan, payment failures. Target: cancelled in the
following month.

Leakage is everywhere here and it is temporal. A `cancellation_reason` field is
obviously fatal. Less obviously: "number of support tickets" counted over a
window that overlaps the prediction month, or a `plan` field that already records
the downgrade they made while cancelling. The discipline is to fix a cutoff date
and use **only** what was knowable before it.

Baseline: predict cancellation for anyone with zero logins in 30 days. It is
often startlingly hard to beat, and if the model does not, the model is not
worth deploying.

Also note question 4: a false positive costs you a discount offer, a false
negative costs you a customer. Those are wildly different numbers, so accuracy is
the wrong metric and the threshold is a business decision, not a modelling one.

**2. Duplicate tickets.** This is unsupervised, or *nearly* — you probably have a
few thousand pairs a human already merged, which makes it weakly supervised and
changes the approach entirely. Ask before assuming.

Task: clustering, or better, retrieval — embed each ticket and find near
neighbours ([Chapter 12](/learn/12-embeddings-and-tabular/)). Framing it as
retrieval rather than clustering is the better call, because it gives you a
ranked list a human can confirm rather than a partition you must trust.

Baseline: TF-IDF cosine similarity. It is thirty years old, takes an afternoon,
and is a genuinely strong baseline on this task.

**3. VAT.** Not a machine learning problem. It is defined in legislation; write
the rules. A model would give you 99.4% accuracy where 100% was free, and would
be unable to explain the 0.6% to an auditor.

The interesting variant: *extracting the line items from a scanned invoice* very
much is a machine learning problem — supervised, and these days a vision-language
model. Note the shape of that: the perception step is learned, the arithmetic
step stays code. Splitting a system along that seam is one of the most useful
design instincts you can develop.

Tomorrow: build a model from nothing, and watch it learn.